# Heart Disease Prediction — Ensemble Notebook

**Competition:** Playground Series S6E2  
**Approach:** CatBoost (base + alt variants) × RealMLP × XGBoost → Meta Ridge blending

| | Score |
|---|---|
| **CV (OOF AUC)** | 0.955724 |
| **Public LB** | 0.95376 |

## 1. Environment & Installs

In [1]:
# Install all required packages in one place
!pip install pytabkit catboost xgboost -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 364.0/364.0 kB 7.2 MB/s eta 0:00:00


## 2. Imports & Global Config

In [2]:
# ── Standard library ──────────────────────────────────────────────────────────
import json
import subprocess
import warnings
import zipfile
from pathlib import Path

# ── Numerical / data ──────────────────────────────────────────────────────────
import numpy as np
import pandas as pd

# ── Visualization ─────────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns

# ── Machine learning ──────────────────────────────────────────────────────────
import torch
from catboost import CatBoostClassifier, Pool
from pytabkit import RealMLP_TD_Classifier
from scipy.stats import rankdata, spearmanr
from sklearn.linear_model import Ridge
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb

warnings.filterwarnings('ignore')
sns.set_theme(style="whitegrid")

# ── Global config ─────────────────────────────────────────────────────────────
CFG = dict(
    comp_slug       = "playground-series-s6e2",
    target_col      = "Heart Disease",
    id_col          = "id",
    n_folds         = 5,
    random_state    = 42,
    repeated_seeds  = [42, 2024, 2025],
    ridge_alphas    = [10.0, 30.0, 100.0, 300.0],
    # Paths
    kaggle_comp_dir = Path("/kaggle/input/competitions/playground-series-s6e2"),
    kaggle_ext_path = Path("/kaggle/input/datasets/neurocipher/heartdisease/Heart_Disease_Prediction.csv"),
    local_data_dir  = Path("data/raw"),
    need_files      = ["train.csv", "test.csv", "sample_submission.csv"],
)

# Set all random seeds
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
torch.manual_seed(CFG['random_state'])
np.random.seed(CFG['random_state'])
if DEVICE == 'cuda':
    torch.cuda.manual_seed_all(CFG['random_state'])
    torch.set_float32_matmul_precision('high')

CFG['local_data_dir'].mkdir(parents=True, exist_ok=True)
print(f"Device: {DEVICE}")

Device: cuda


## 3. Utilities

In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Data / Kaggle helpers
# ─────────────────────────────────────────────────────────────────────────────

def _run(cmd: list[str]) -> None:
    """Run a subprocess command; raises on non-zero exit."""
    subprocess.run(cmd, capture_output=True, text=True, check=True)


def _ensure_kaggle_json_colab(dst: Path = Path("/content/kaggle.json")) -> Path:
    """Prompt an upload dialog in Colab if kaggle.json is missing; otherwise require it exists."""
    if dst.exists():
        print("Found:", dst)
        return dst
    try:
        from google.colab import files  # type: ignore
    except ImportError:
        raise FileNotFoundError(
            f"{dst} not found. Place kaggle.json at {dst} or set up credentials another way."
        )
    print("Upload your kaggle.json (Kaggle → Account → API → Create New Token)")
    uploaded = files.upload()
    cand = next((n for n in uploaded if n.endswith(".json")), None)
    if cand is None:
        raise FileNotFoundError("Upload failed: no .json file received.")
    Path(cand).rename(dst)
    print("Saved to:", dst)
    return dst


def _install_kaggle_json(src: Path) -> None:
    """Copy kaggle.json → ~/.kaggle/kaggle.json with chmod 600."""
    if not src.exists():
        raise FileNotFoundError(f"{src} not found.")
    dst_dir = Path.home() / ".kaggle"
    dst_dir.mkdir(parents=True, exist_ok=True)
    dst = dst_dir / "kaggle.json"
    dst.write_bytes(src.read_bytes())
    try:
        dst.chmod(0o600)
    except Exception:
        pass
    cfg = json.loads(dst.read_text())
    if "username" not in cfg or "key" not in cfg:
        raise ValueError("kaggle.json is missing 'username' or 'key'.")
    print(f"Installed kaggle.json for user: {cfg['username']}")


def _local_data_ready(data_dir: Path) -> bool:
    return all((data_dir / f).exists() for f in CFG['need_files'])


def _download_competition(data_dir: Path) -> None:
    """Download and unzip competition data via kaggle CLI."""
    _run(["kaggle", "config", "view"])
    _run(["kaggle", "competitions", "download", "-c", CFG['comp_slug'], "-p", str(data_dir), "--force"])
    zips = list(data_dir.glob("*.zip"))
    if not zips:
        raise FileNotFoundError(f"No zip found in {data_dir} after download.")
    for zp in zips:
        with zipfile.ZipFile(zp, "r") as zf:
            zf.extractall(data_dir)
        print("Unzipped:", zp.name)
    if not _local_data_ready(data_dir):
        missing = [f for f in CFG['need_files'] if not (data_dir / f).exists()]
        raise FileNotFoundError(f"Missing after download: {missing}")


# ─────────────────────────────────────────────────────────────────────────────
# Preprocessing helpers
# ─────────────────────────────────────────────────────────────────────────────

def encode_target_strict(y: pd.Series) -> pd.Series:
    """Map common string target labels to {0, 1}. Raises on unknown values."""
    mapping_candidates = [
        {"No": 0, "Yes": 1}, {"N": 0, "Y": 1},
        {"Negative": 0, "Positive": 1}, {"Absent": 0, "Present": 1},
        {"Absence": 0, "Presence": 1}, {0: 0, 1: 1}, {"0": 0, "1": 1},
    ]
    uniq = set(pd.Series(y).dropna().unique().tolist())
    for mp in mapping_candidates:
        if uniq.issubset(set(mp.keys())):
            return pd.Series(y).map(mp).astype("int8")
    raise ValueError(f"Unknown target labels: {sorted(list(uniq))}")


def check_data_quality(df: pd.DataFrame, name: str = "Dataset") -> None:
    """Print a concise data quality summary."""
    cols = [c for c in df.columns if c != 'id']
    dupes = df.duplicated(subset=cols).sum()
    nans  = df.isnull().sum().sum()
    print(f"[{name}] rows={len(df)}, duplicates={dupes}, total_NaN={nans}")
    nan_series = df.isnull().sum()
    if nans > 0:
        print(nan_series[nan_series > 0].to_string())


def analyze_feature_types(df: pd.DataFrame) -> pd.DataFrame:
    """Return a DataFrame summarising unique value counts and heuristic types."""
    rows = []
    for col in df.columns:
        if col == 'id':
            continue
        n_unique = df[col].nunique()
        is_num   = pd.api.types.is_numeric_dtype(df[col])
        rows.append({
            'Feature': col,
            'Unique': n_unique,
            'DType': df[col].dtype,
            'Heuristic': 'Categorical' if n_unique < 25 else 'Continuous',
            'Mean': float(pd.to_numeric(df[col], errors='coerce').mean()) if is_num else float('nan'),
            'Std':  float(pd.to_numeric(df[col], errors='coerce').std())  if is_num else float('nan'),
        })
    return pd.DataFrame(rows).sort_values('Unique')


# ─────────────────────────────────────────────────────────────────────────────
# Feature engineering helpers
# ─────────────────────────────────────────────────────────────────────────────

def add_external_target_stats(
    df: pd.DataFrame,
    original_df: pd.DataFrame | None,
    base_features: list[str],
    target_col: str,
) -> pd.DataFrame:
    """
    Merge group-wise target statistics from the external/original dataset into df.
    Safe in Playground competitions because original labels are not the competition labels.
    """
    if original_df is None:
        return df.copy()
    out = df.copy()
    n = len(out)
    global_mean   = float(original_df[target_col].mean())
    global_median = float(original_df[target_col].median())
    for col in base_features:
        if col not in original_df.columns:
            continue
        stats = (
            original_df.groupby(col)[target_col]
            .agg(["mean", "median", "std", "skew", "count"])
            .reset_index()
        )
        stats.columns = [col] + [f"orig_{col}_{s}" for s in ["mean", "median", "std", "skew", "count"]]
        out = out.merge(stats, on=col, how="left")
        assert len(out) == n, f"Merge expanded rows for column {col}"
        out = out.fillna({
            f"orig_{col}_mean":   global_mean,
            f"orig_{col}_median": global_median,
            f"orig_{col}_std":    0.0,
            f"orig_{col}_skew":   0.0,
            f"orig_{col}_count":  0.0,
        })
    return out


def to_xgb_matrix(X_df: pd.DataFrame, X_test_df: pd.DataFrame, cat_cols: list[str]):
    """Convert category columns to integer codes for XGBoost; aligns train/test categories."""
    X_tr, X_te = X_df.copy(), X_test_df.copy()
    for c in cat_cols:
        cats = pd.Categorical(pd.concat([X_tr[c], X_te[c]], ignore_index=True)).categories
        X_tr[c] = pd.Categorical(X_tr[c], categories=cats).codes.astype("int32")
        X_te[c] = pd.Categorical(X_te[c], categories=cats).codes.astype("int32")
    for c in X_tr.columns:
        X_tr[c] = pd.to_numeric(X_tr[c], errors="coerce").astype("float32")
        X_te[c] = pd.to_numeric(X_te[c], errors="coerce").astype("float32")
    return X_tr, X_te


# ─────────────────────────────────────────────────────────────────────────────
# Ensembling / meta-learning helpers
# ─────────────────────────────────────────────────────────────────────────────

def rank01(x: np.ndarray) -> np.ndarray:
    """Map array to [0, 1] via average-rank normalisation (scipy rankdata)."""
    x = np.asarray(x, dtype=np.float64)
    r = rankdata(x, method="average")
    return (r - 0.5) / len(r)


def minmax01(x: np.ndarray) -> np.ndarray:
    """Min-max scale array to [0, 1]. Returns 0.5-filled array if constant."""
    x  = np.asarray(x, dtype=np.float64)
    mn, mx = x.min(), x.max()
    if mx - mn < 1e-12:
        return np.full_like(x, 0.5)
    return (x - mn) / (mx - mn)


def zscore(x: np.ndarray) -> np.ndarray:
    """Standardise array to zero mean / unit variance."""
    x  = np.asarray(x, dtype=np.float64)
    sd = x.std()
    return (x - x.mean()) / sd if sd > 1e-12 else np.zeros_like(x)


def safe_array(x, n: int, fill_value: float = 0.0, name: str = "arr") -> np.ndarray:
    """Return array of length n; fills with fill_value if x is None."""
    if x is None:
        return np.full(n, fill_value, dtype="float64")
    arr = np.asarray(x, dtype="float64")
    if len(arr) != n:
        raise ValueError(f"{name} length mismatch: expected {n}, got {len(arr)}")
    return arr


def build_meta_features(
    oof_dict: dict[str, np.ndarray],
    test_dict: dict[str, np.ndarray],
    model_order: list[str],
    feature_mode: str = "rank+prob",
):
    """
    Build meta-feature matrices from base-model OOF/test predictions.

    feature_mode options: "rank" | "prob" | "rank+prob" | "rank+z"
    Returns: (X_meta_oof, X_meta_test, feature_names)
    """
    Xo, Xt, names = [], [], []
    if "rank" in feature_mode:
        for m in model_order:
            Xo.append(rank01(oof_dict[m]))
            Xt.append(rank01(test_dict[m]))
            names.append(f"{m}_rank")
    if "prob" in feature_mode:
        for m in model_order:
            Xo.append(np.asarray(oof_dict[m], dtype=np.float64))
            Xt.append(np.asarray(test_dict[m], dtype=np.float64))
            names.append(f"{m}_prob")
    if "z" in feature_mode:
        for m in model_order:
            Xo.append(zscore(oof_dict[m]))
            Xt.append(zscore(test_dict[m]))
            names.append(f"{m}_z")
    return np.column_stack(Xo), np.column_stack(Xt), names


def fit_meta_ridge(
    y: np.ndarray,
    oof_dict: dict[str, np.ndarray],
    test_dict: dict[str, np.ndarray],
    model_order: list[str],
    alpha_grid: list[float] = None,
    feature_mode: str = "rank+prob",
    n_splits: int = 5,
    seed: int = 42,
) -> dict:
    """
    Fit a stacked Ridge meta-learner via StratifiedKFold CV.
    Alpha is selected fold-by-fold on the val set.

    Returns a dict with keys:
      meta_oof_pred, meta_test_pred, meta_test_pred_01,
      meta_oof_auc, fold_info, feature_names
    """
    if alpha_grid is None:
        alpha_grid = CFG['ridge_alphas']
    y = np.asarray(y, dtype=np.int64)
    X_oof, X_test, feat_names = build_meta_features(oof_dict, test_dict, model_order, feature_mode)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)

    meta_oof  = np.zeros(len(y), dtype=np.float64)
    test_preds_per_fold = []
    fold_info = []

    for fold, (tr_idx, va_idx) in enumerate(skf.split(X_oof, y), start=1):
        X_tr, X_va = X_oof[tr_idx], X_oof[va_idx]
        y_tr, y_va = y[tr_idx], y[va_idx]

        best_auc, best_alpha, best_model, best_va_pred = -1.0, None, None, None
        for alpha in alpha_grid:
            mdl = Ridge(alpha=alpha, fit_intercept=True)
            mdl.fit(X_tr, y_tr)
            va_pred = mdl.predict(X_va)
            auc = roc_auc_score(y_va, va_pred)
            if auc > best_auc:
                best_auc, best_alpha, best_model, best_va_pred = auc, alpha, mdl, va_pred

        meta_oof[va_idx] = best_va_pred
        test_preds_per_fold.append(best_model.predict(X_test))
        fold_info.append({"fold": fold, "best_alpha": best_alpha, "val_auc": best_auc})
        print(f"  [Fold {fold}] AUC={best_auc:.6f}, alpha={best_alpha}")

    meta_test = np.mean(test_preds_per_fold, axis=0)
    oof_auc   = roc_auc_score(y, meta_oof)
    print(f"  [OOF AUC] {oof_auc:.6f}  (feature_mode={feature_mode})")

    return {
        "meta_oof_pred":    meta_oof,
        "meta_test_pred":   meta_test,
        "meta_test_pred_01": minmax01(meta_test),  # clipped to [0,1] for submission readability
        "meta_oof_auc":     oof_auc,
        "fold_info":        pd.DataFrame(fold_info),
        "feature_names":    feat_names,
    }

## 4. Data Access (Kaggle / Colab / Local)

In [4]:
kaggle_dir = CFG['kaggle_comp_dir']
local_dir  = CFG['local_data_dir']

if kaggle_dir.exists():
    data_dir = kaggle_dir
    print("[Kaggle] Using mounted competition data:", data_dir)
else:
    data_dir = local_dir
    if _local_data_ready(data_dir):
        print("[Local] Data already present:", data_dir)
    else:
        print("[Download] Fetching via Kaggle API...")
        kaggle_json = _ensure_kaggle_json_colab(Path("/content/kaggle.json"))
        _install_kaggle_json(kaggle_json)
        _download_competition(data_dir)
        print("Download complete.")

[Kaggle] Using mounted competition data: /kaggle/input/competitions/playground-series-s6e2


## 5. Data Loading & EDA

In [5]:
# ── Load competition files ─────────────────────────────────────────────────────
for fname in CFG['need_files']:
    if not (data_dir / fname).exists():
        raise FileNotFoundError(f"Required file missing: {data_dir / fname}")

train = pd.read_csv(data_dir / "train.csv")
test  = pd.read_csv(data_dir / "test.csv")
sub   = pd.read_csv(data_dir / "sample_submission.csv")

# Optional external dataset (Kaggle-only)
ext_path = CFG['kaggle_ext_path']
original = pd.read_csv(ext_path) if ext_path.exists() else None

print(f"train: {train.shape}, test: {test.shape}, sub: {sub.shape}")
print(f"external: {'loaded' if original is not None else 'not available'}")
print("Columns only in train:", sorted(set(train.columns) - set(test.columns)))
print("Columns only in test: ", sorted(set(test.columns) - set(train.columns)))

train: (630000, 15), test: (270000, 14), sub: (270000, 2)
external: not available
Columns only in train: ['Heart Disease']
Columns only in test:  []


In [6]:
# ── Quick data quality check ───────────────────────────────────────────────────
check_data_quality(train, "Train")
check_data_quality(test,  "Test")

# ── Feature cardinality overview ───────────────────────────────────────────────
analyze_feature_types(train)

[Train] rows=630000, duplicates=0, total_NaN=0
[Test] rows=270000, duplicates=0, total_NaN=0


,Feature,Unique,DType,Heuristic,Mean,Std
1,Sex,2,int64,Categorical,0.714735,0.451541
5,FBS over 120,2,int64,Categorical,0.079987,0.271274
13,Heart Disease,2,object,Categorical,NaN,NaN
8,Exercise angina,2,int64,Categorical,0.273725,0.445870
10,Slope of ST,3,int64,Categorical,1.455871,0.545192
12,Thallium,3,int64,Categorical,4.618873,1.950007
6,EKG results,3,int64,Categorical,0.981660,0.998783
2,Chest pain type,4,int64,Categorical,3.312752,0.851615
11,Number of vessels fluro,4,int64,Categorical,0.451040,0.798549
0,Age,42,int64,Continuous,54.136706,8.256301


In [7]:
# ── Target distribution ────────────────────────────────────────────────────────
display(train.head())

print(train[CFG['target_col']].value_counts(normalize=True).rename("proportion").to_frame())

,id,Age,Sex,Chest pain type,BP,Cholesterol,FBS over 120,EKG results,Max HR,Exercise angina,ST depression,Slope of ST,Number of vessels fluro,Thallium,Heart Disease
0,0,58,1,4,152,239,0,0,158,1,3.6,2,2,7,Presence
1,1,52,1,1,125,325,0,2,171,0,0.0,1,0,3,Absence
2,2,56,0,2,160,188,0,2,151,0,0.0,1,0,3,Absence
3,3,44,0,3,134,229,0,2,150,0,1.0,2,0,3,Absence
4,4,58,1,4,140,234,0,2,125,1,3.8,2,3,3,Presence


               proportion
Heart Disease            
Absence           0.55166
Presence          0.44834


## 6. Preprocessing & Feature Engineering

In [8]:
TARGET_COL = CFG['target_col']
ID_COL     = CFG['id_col']
META_COLS  = [TARGET_COL, ID_COL, "is_external"]

# ── 1. Encode target ──────────────────────────────────────────────────────────
train_comp = train.copy()
if not pd.api.types.is_numeric_dtype(train_comp[TARGET_COL]):
    le = LabelEncoder()
    train_comp[TARGET_COL] = le.fit_transform(train_comp[TARGET_COL])
    if original is not None and TARGET_COL in original.columns:
        original[TARGET_COL] = le.transform(original[TARGET_COL])

# ── 2. Align and append external data ─────────────────────────────────────────
if original is not None:
    if TARGET_COL not in original.columns:
        raise ValueError("External dataset missing target column.")
    if not pd.api.types.is_numeric_dtype(original[TARGET_COL]):
        original[TARGET_COL] = encode_target_strict(original[TARGET_COL])

    orig_aligned = original.copy()
    for col in train_comp.columns:
        if col not in orig_aligned.columns:
            orig_aligned[col] = np.nan
    if ID_COL not in orig_aligned.columns:
        orig_aligned[ID_COL] = -(np.arange(len(orig_aligned)) + 1)
    orig_aligned = orig_aligned[train_comp.columns]

    train_full = pd.concat([train_comp, orig_aligned], ignore_index=True)
    train_full["is_external"] = [0] * len(train_comp) + [1] * len(orig_aligned)
else:
    train_full = train_comp.copy()
    train_full["is_external"] = 0

train = train_full

print(f"Combined train: {train.shape}")
print(train[['is_external', TARGET_COL]].value_counts().sort_index())

Combined train: (630000, 16)
is_external  Heart Disease
0            0                347546
             1                282454
Name: count, dtype: int64


In [9]:
# ── 3. Define categorical columns ─────────────────────────────────────────────
# Canonical semantic categoricals for S6E2
CANONICAL_CAT = {
    "Sex", "Chest pain type", "FBS over 120", "EKG results",
    "Exercise angina", "Slope of ST", "Number of vessels fluro", "Thallium",
}
BASE_FEATURES = [c for c in train.columns if c not in META_COLS]
cat_cols = [c for c in BASE_FEATURES if c in CANONICAL_CAT]

# ── 4. Feature engineering: external target statistics ────────────────────────
train_fe = add_external_target_stats(train, original, BASE_FEATURES, TARGET_COL)
test_fe  = add_external_target_stats(test,  original, BASE_FEATURES, TARGET_COL)

# ── 5. Build X / y / X_test ───────────────────────────────────────────────────
X = train_fe.drop(columns=[c for c in META_COLS if c in train_fe.columns])
y = train_fe[TARGET_COL].astype("int8")
X_test = test_fe.drop(columns=[c for c in [ID_COL] if c in test_fe.columns])

# All columns treated as categorical strings (matches RealMLP CV notebook approach)
for col in X.columns:
    X[col]      = X[col].astype(str).astype("category")
    X_test[col] = X_test[col].astype(str).astype("category")

# cat_cols for models that need explicit category lists
cat_cols = list(X.columns)

print(f"X: {X.shape}  |  X_test: {X_test.shape}")

X: (630000, 13)  |  X_test: (270000, 13)


## 7. Validation Setup

In [10]:
N_FOLDS = CFG['n_folds']
RANDOM_STATE = CFG['random_state']

# Competition rows only (exclude external data from CV)
comp_mask = (train["is_external"] == 0).values

X_comp = X.loc[comp_mask].reset_index(drop=True)
y_comp = y.loc[comp_mask].reset_index(drop=True)
comp_ids = train.loc[comp_mask, ID_COL].reset_index(drop=True)

X_test_comp = X_test.copy()
test_ids    = test[ID_COL].reset_index(drop=True)

# Shared CV splitter (all base models use this)
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)

print(f"X_comp: {X_comp.shape}  |  y_comp: {y_comp.shape}  |  X_test_comp: {X_test_comp.shape}")
print(f"CV: {N_FOLDS}-fold stratified, seed={RANDOM_STATE}")

X_comp: (630000, 13)  |  y_comp: (630000,)  |  X_test_comp: (270000, 13)
CV: 5-fold stratified, seed=42


## 8. Base Model Training

### 8a. CatBoost (Base)

In [11]:
cat_features_idx = [X_comp.columns.get_loc(c) for c in cat_cols]

oof_cat  = np.zeros(len(X_comp), dtype="float32")
test_cat = np.zeros(len(X_test_comp), dtype="float32")

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_comp, y_comp), start=1):
    print(f"[CatBoost] Fold {fold}/{N_FOLDS}")
    X_tr, X_val = X_comp.iloc[tr_idx], X_comp.iloc[val_idx]
    y_tr, y_val = y_comp.iloc[tr_idx], y_comp.iloc[val_idx]

    train_pool = Pool(X_tr, label=y_tr, cat_features=cat_features_idx)
    valid_pool = Pool(X_val, label=y_val, cat_features=cat_features_idx)
    test_pool  = Pool(X_test_comp, cat_features=cat_features_idx)

    model = CatBoostClassifier(
        loss_function="Logloss", eval_metric="AUC",
        learning_rate=0.03, depth=6, l2_leaf_reg=3.0,
        random_seed=RANDOM_STATE + fold,
        iterations=4000, od_type="Iter", od_wait=200,
        verbose=200,
        task_type="GPU" if DEVICE == "cuda" else "CPU",
    )
    model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

    oof_cat[val_idx]  = model.predict_proba(valid_pool)[:, 1].astype("float32")
    test_cat         += (model.predict_proba(test_pool)[:, 1] / N_FOLDS).astype("float32")

cat_oof_auc = roc_auc_score(y_comp, oof_cat)
print(f"\n[CatBoost Base] OOF AUC = {cat_oof_auc:.6f}")

[CatBoost] Fold 1/5


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9406781	best: 0.9406781 (0)	total: 141ms	remaining: 9m 23s
200:	test: 0.9557329	best: 0.9557329 (200)	total: 9.22s	remaining: 2m 54s
400:	test: 0.9559670	best: 0.9559671 (399)	total: 18.3s	remaining: 2m 44s
600:	test: 0.9560025	best: 0.9560025 (600)	total: 27.4s	remaining: 2m 34s
800:	test: 0.9560089	best: 0.9560108 (742)	total: 36.5s	remaining: 2m 25s
1000:	test: 0.9560131	best: 0.9560134 (860)	total: 45.6s	remaining: 2m 16s
1200:	test: 0.9560172	best: 0.9560177 (1196)	total: 54.8s	remaining: 2m 7s
1400:	test: 0.9560163	best: 0.9560193 (1241)	total: 1m 4s	remaining: 1m 59s
1600:	test: 0.9560165	best: 0.9560211 (1459)	total: 1m 13s	remaining: 1m 50s
bestTest = 0.9560210705
bestIteration = 1459
Shrink model to first 1460 iterations.
[CatBoost] Fold 2/5


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9390359	best: 0.9390359 (0)	total: 52.5ms	remaining: 3m 29s
200:	test: 0.9545804	best: 0.9545804 (200)	total: 9.33s	remaining: 2m 56s
400:	test: 0.9548081	best: 0.9548081 (400)	total: 18.5s	remaining: 2m 46s
600:	test: 0.9548547	best: 0.9548548 (595)	total: 27.6s	remaining: 2m 36s
800:	test: 0.9548821	best: 0.9548821 (799)	total: 36.8s	remaining: 2m 27s
1000:	test: 0.9548910	best: 0.9548916 (983)	total: 46s	remaining: 2m 17s
1200:	test: 0.9548910	best: 0.9548928 (1023)	total: 55.2s	remaining: 2m 8s
bestTest = 0.9548927546
bestIteration = 1023
Shrink model to first 1024 iterations.
[CatBoost] Fold 3/5


...

0:	test: 0.9394967	best: 0.9394967 (0)	total: 60.9ms	remaining: 4m 3s
200:	test: 0.9553767	best: 0.9553767 (200)	total: 9.23s	remaining: 2m 54s
400:	test: 0.9556374	best: 0.9556376 (399)	total: 18.6s	remaining: 2m 47s
600:	test: 0.9556866	best: 0.9556866 (600)	total: 27.9s	remaining: 2m 37s
800:	test: 0.9557022	best: 0.9557030 (794)	total: 37.1s	remaining: 2m 28s
bestTest = 0.9557030201
bestIteration = 794
Shrink model to first 795 iterations.
[CatBoost] Fold 4/5


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9393728	best: 0.9393728 (0)	total: 58.2ms	remaining: 3m 52s
200:	test: 0.9549214	best: 0.9549214 (200)	total: 9.06s	remaining: 2m 51s
400:	test: 0.9552022	best: 0.9552024 (397)	total: 18.4s	remaining: 2m 45s
600:	test: 0.9552596	best: 0.9552600 (599)	total: 27.7s	remaining: 2m 36s
800:	test: 0.9552770	best: 0.9552775 (797)	total: 36.8s	remaining: 2m 27s
1000:	test: 0.9552888	best: 0.9552889 (997)	total: 46.1s	remaining: 2m 18s
1200:	test: 0.9552946	best: 0.9552953 (1152)	total: 55.2s	remaining: 2m 8s
1400:	test: 0.9552990	best: 0.9552990 (1400)	total: 1m 4s	remaining: 1m 59s
1600:	test: 0.9553022	best: 0.9553043 (1550)	total: 1m 13s	remaining: 1m 50s
bestTest = 0.9553043246
bestIteration = 1550
Shrink model to first 1551 iterations.
[CatBoost] Fold 5/5


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9400386	best: 0.9400386 (0)	total: 51.5ms	remaining: 3m 25s
200:	test: 0.9558381	best: 0.9558381 (200)	total: 9.09s	remaining: 2m 51s
400:	test: 0.9560779	best: 0.9560779 (400)	total: 18.3s	remaining: 2m 44s
600:	test: 0.9561254	best: 0.9561254 (600)	total: 27.4s	remaining: 2m 35s
800:	test: 0.9561440	best: 0.9561445 (788)	total: 36.6s	remaining: 2m 26s
1000:	test: 0.9561525	best: 0.9561527 (958)	total: 45.8s	remaining: 2m 17s
1200:	test: 0.9561537	best: 0.9561573 (1134)	total: 55.1s	remaining: 2m 8s
bestTest = 0.9561572671
bestIteration = 1134
Shrink model to first 1135 iterations.

[CatBoost Base] OOF AUC = 0.955611


### 8b. CatBoost ALT (diverse hyperparameters)

In [12]:
oof_cat_alt  = np.zeros(len(X_comp), dtype="float32")
test_cat_alt = np.zeros(len(X_test_comp), dtype="float32")

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_comp, y_comp), start=1):
    print(f"[CatBoost ALT] Fold {fold}/{N_FOLDS}")
    X_tr, X_val = X_comp.iloc[tr_idx], X_comp.iloc[val_idx]
    y_tr, y_val = y_comp.iloc[tr_idx], y_comp.iloc[val_idx]

    train_pool = Pool(X_tr, label=y_tr, cat_features=cat_features_idx)
    valid_pool = Pool(X_val, label=y_val, cat_features=cat_features_idx)
    test_pool  = Pool(X_test_comp, cat_features=cat_features_idx)

    model_alt = CatBoostClassifier(
        loss_function="Logloss", eval_metric="AUC",
        iterations=5000, learning_rate=0.02, depth=5,
        l2_leaf_reg=8.0, random_strength=1.5, bagging_temperature=1.0,
        od_type="Iter", od_wait=250,
        random_seed=RANDOM_STATE + 100 + fold,
        verbose=200,
        task_type="GPU" if DEVICE == "cuda" else "CPU",
    )
    model_alt.fit(train_pool, eval_set=valid_pool, use_best_model=True)

    oof_cat_alt[val_idx]  = model_alt.predict_proba(valid_pool)[:, 1].astype("float32")
    test_cat_alt         += (model_alt.predict_proba(test_pool)[:, 1] / N_FOLDS).astype("float32")

cat_alt_auc = roc_auc_score(y_comp, oof_cat_alt)
print(f"\n[CatBoost ALT] OOF AUC = {cat_alt_auc:.6f}")

[CatBoost ALT] Fold 1/5


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9343147	best: 0.9343147 (0)	total: 45.5ms	remaining: 3m 47s
200:	test: 0.9550449	best: 0.9550449 (200)	total: 7.78s	remaining: 3m 5s
400:	test: 0.9558069	best: 0.9558069 (400)	total: 15.8s	remaining: 3m 1s
600:	test: 0.9559491	best: 0.9559491 (600)	total: 23.8s	remaining: 2m 54s
800:	test: 0.9559973	best: 0.9559974 (799)	total: 31.7s	remaining: 2m 45s
1000:	test: 0.9560152	best: 0.9560152 (1000)	total: 39.5s	remaining: 2m 37s
1200:	test: 0.9560263	best: 0.9560263 (1195)	total: 47.3s	remaining: 2m 29s
1400:	test: 0.9560339	best: 0.9560339 (1397)	total: 55.3s	remaining: 2m 22s
1600:	test: 0.9560370	best: 0.9560376 (1594)	total: 1m 3s	remaining: 2m 13s
1800:	test: 0.9560412	best: 0.9560413 (1798)	total: 1m 10s	remaining: 2m 5s
2000:	test: 0.9560435	best: 0.9560440 (1967)	total: 1m 18s	remaining: 1m 57s
2200:	test: 0.9560440	best: 0.9560441 (2183)	total: 1m 26s	remaining: 1m 49s
2400:	test: 0.9560442	best: 0.9560446 (2380)	total: 1m 34s	remaining: 1m 41s
2600:	test: 0.9560467	be

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9301895	best: 0.9301895 (0)	total: 44.9ms	remaining: 3m 44s
200:	test: 0.9538659	best: 0.9538659 (200)	total: 7.74s	remaining: 3m 4s
400:	test: 0.9546406	best: 0.9546406 (400)	total: 15.7s	remaining: 2m 59s
600:	test: 0.9547853	best: 0.9547853 (600)	total: 23.6s	remaining: 2m 52s
800:	test: 0.9548386	best: 0.9548386 (799)	total: 31.4s	remaining: 2m 44s
1000:	test: 0.9548615	best: 0.9548615 (1000)	total: 39.3s	remaining: 2m 36s
1200:	test: 0.9548770	best: 0.9548770 (1198)	total: 47.1s	remaining: 2m 28s
1400:	test: 0.9548903	best: 0.9548905 (1395)	total: 54.9s	remaining: 2m 21s
1600:	test: 0.9548976	best: 0.9548976 (1600)	total: 1m 2s	remaining: 2m 13s
1800:	test: 0.9549016	best: 0.9549022 (1759)	total: 1m 10s	remaining: 2m 5s
2000:	test: 0.9549052	best: 0.9549056 (1973)	total: 1m 18s	remaining: 1m 57s
2200:	test: 0.9549087	best: 0.9549089 (2189)	total: 1m 26s	remaining: 1m 49s
2400:	test: 0.9549125	best: 0.9549135 (2364)	total: 1m 34s	remaining: 1m 41s
2600:	test: 0.9549130	b

...

0:	test: 0.9298628	best: 0.9298628 (0)	total: 44.9ms	remaining: 3m 44s
200:	test: 0.9545889	best: 0.9545889 (200)	total: 7.83s	remaining: 3m 6s
400:	test: 0.9554281	best: 0.9554281 (400)	total: 15.9s	remaining: 3m 1s
600:	test: 0.9555813	best: 0.9555813 (600)	total: 23.9s	remaining: 2m 54s
800:	test: 0.9556395	best: 0.9556395 (800)	total: 31.8s	remaining: 2m 46s
1000:	test: 0.9556614	best: 0.9556616 (988)	total: 39.5s	remaining: 2m 37s
1200:	test: 0.9556744	best: 0.9556748 (1195)	total: 47.3s	remaining: 2m 29s
1400:	test: 0.9556832	best: 0.9556832 (1400)	total: 55.2s	remaining: 2m 21s
1600:	test: 0.9556900	best: 0.9556903 (1597)	total: 1m 2s	remaining: 2m 13s
1800:	test: 0.9556968	best: 0.9556969 (1793)	total: 1m 10s	remaining: 2m 5s
2000:	test: 0.9556995	best: 0.9556996 (1995)	total: 1m 18s	remaining: 1m 57s
2200:	test: 0.9557001	best: 0.9557012 (2165)	total: 1m 26s	remaining: 1m 49s
2400:	test: 0.9557015	best: 0.9557021 (2323)	total: 1m 34s	remaining: 1m 42s
2600:	test: 0.9557050	bes

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9326176	best: 0.9326176 (0)	total: 46.1ms	remaining: 3m 50s
200:	test: 0.9542201	best: 0.9542201 (200)	total: 7.87s	remaining: 3m 7s
400:	test: 0.9550062	best: 0.9550062 (400)	total: 15.9s	remaining: 3m 1s
600:	test: 0.9551830	best: 0.9551830 (600)	total: 23.8s	remaining: 2m 54s
800:	test: 0.9552432	best: 0.9552432 (799)	total: 31.5s	remaining: 2m 45s
1000:	test: 0.9552718	best: 0.9552719 (988)	total: 39.4s	remaining: 2m 37s
1200:	test: 0.9552890	best: 0.9552891 (1198)	total: 47.2s	remaining: 2m 29s
1400:	test: 0.9553038	best: 0.9553038 (1400)	total: 55.1s	remaining: 2m 21s
1600:	test: 0.9553143	best: 0.9553143 (1600)	total: 1m 2s	remaining: 2m 13s
1800:	test: 0.9553213	best: 0.9553217 (1781)	total: 1m 10s	remaining: 2m 5s
2000:	test: 0.9553246	best: 0.9553249 (1992)	total: 1m 18s	remaining: 1m 58s
2200:	test: 0.9553292	best: 0.9553292 (2198)	total: 1m 26s	remaining: 1m 50s
2400:	test: 0.9553323	best: 0.9553324 (2395)	total: 1m 34s	remaining: 1m 42s
2600:	test: 0.9553363	bes

Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9336584	best: 0.9336584 (0)	total: 45.9ms	remaining: 3m 49s
200:	test: 0.9551258	best: 0.9551258 (200)	total: 7.96s	remaining: 3m 10s
400:	test: 0.9559013	best: 0.9559013 (400)	total: 16s	remaining: 3m 3s
600:	test: 0.9560687	best: 0.9560687 (600)	total: 23.9s	remaining: 2m 54s
800:	test: 0.9561200	best: 0.9561200 (800)	total: 31.8s	remaining: 2m 46s
1000:	test: 0.9561434	best: 0.9561434 (1000)	total: 39.6s	remaining: 2m 38s
1200:	test: 0.9561566	best: 0.9561566 (1195)	total: 47.4s	remaining: 2m 29s
1400:	test: 0.9561670	best: 0.9561670 (1400)	total: 55.4s	remaining: 2m 22s
1600:	test: 0.9561736	best: 0.9561737 (1593)	total: 1m 3s	remaining: 2m 14s
1800:	test: 0.9561786	best: 0.9561789 (1792)	total: 1m 11s	remaining: 2m 6s
2000:	test: 0.9561800	best: 0.9561800 (2000)	total: 1m 19s	remaining: 1m 58s
2200:	test: 0.9561840	best: 0.9561840 (2200)	total: 1m 27s	remaining: 1m 50s
2400:	test: 0.9561855	best: 0.9561865 (2337)	total: 1m 34s	remaining: 1m 42s
bestTest = 0.956186533
be

### 8c. CatBoost ALT2 (diversity-focused, multi-seed ensemble)

In [13]:
CAT_ALT2_PARAMS = {
    "loss_function": "Logloss", "eval_metric": "AUC",
    "learning_rate": 0.03, "depth": 8, "l2_leaf_reg": 8.0,
    "random_strength": 1.5, "rsm": 0.80,   # <- CPU時のみ有効にする
    "bootstrap_type": "Bernoulli", "subsample": 0.85,
    "iterations": 4000, "od_type": "Iter", "od_wait": 200,
    "verbose": 200,
    "task_type": "GPU" if DEVICE == "cuda" else "CPU",
}
CAT_ALT2_SEEDS = [42, 2024]

oof_cat_alt2  = np.zeros(len(X_comp), dtype="float32")
test_cat_alt2 = np.zeros(len(X_test_comp), dtype="float32")

for s_idx, seed in enumerate(CAT_ALT2_SEEDS, start=1):
    print(f"\n===== [cat_alt2] seed {seed} ({s_idx}/{len(CAT_ALT2_SEEDS)}) =====")
    oof_seed  = np.zeros(len(X_comp), dtype="float32")
    test_seed = np.zeros(len(X_test_comp), dtype="float32")
    skf_local = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)

    for fold, (tr_idx, val_idx) in enumerate(skf_local.split(X_comp, y_comp), start=1):
        print(f"[cat_alt2] Fold {fold}/{N_FOLDS}")
        X_tr, X_val = X_comp.iloc[tr_idx], X_comp.iloc[val_idx]
        y_tr, y_val = y_comp.iloc[tr_idx], y_comp.iloc[val_idx]

        train_pool = Pool(X_tr, label=y_tr, cat_features=cat_features_idx)
        valid_pool = Pool(X_val, label=y_val, cat_features=cat_features_idx)
        test_pool  = Pool(X_test_comp, cat_features=cat_features_idx)

        params = {**CAT_ALT2_PARAMS, "random_seed": seed + fold}

        # GPU + Logloss では rsm は使えない
        if params.get("task_type") == "GPU":
            params.pop("rsm", None)

        model = CatBoostClassifier(**params)
        model.fit(train_pool, eval_set=valid_pool, use_best_model=True)

        oof_seed[val_idx]  = model.predict_proba(valid_pool)[:, 1].astype("float32")
        test_seed         += (model.predict_proba(test_pool)[:, 1] / N_FOLDS).astype("float32")

    seed_auc = roc_auc_score(y_comp, oof_seed)
    print(f"[cat_alt2][seed={seed}] OOF AUC = {seed_auc:.6f}")
    oof_cat_alt2  += (oof_seed  / len(CAT_ALT2_SEEDS)).astype("float32")
    test_cat_alt2 += (test_seed / len(CAT_ALT2_SEEDS)).astype("float32")

cat_alt2_auc = roc_auc_score(y_comp, oof_cat_alt2)
print(f"\n[cat_alt2 ensemble] OOF AUC = {cat_alt2_auc:.6f}")


===== [cat_alt2] seed 42 (1/2) =====
[cat_alt2] Fold 1/5


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9456944	best: 0.9456944 (0)	total: 98.1ms	remaining: 6m 32s
200:	test: 0.9558464	best: 0.9558464 (200)	total: 13.1s	remaining: 4m 7s
400:	test: 0.9559806	best: 0.9559809 (399)	total: 26.3s	remaining: 3m 55s
600:	test: 0.9559982	best: 0.9560005 (565)	total: 39.1s	remaining: 3m 41s
800:	test: 0.9560011	best: 0.9560044 (759)	total: 52.2s	remaining: 3m 28s
bestTest = 0.9560044408
bestIteration = 759
Shrink model to first 760 iterations.
[cat_alt2] Fold 2/5


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9445210	best: 0.9445210 (0)	total: 106ms	remaining: 7m 5s
200:	test: 0.9546804	best: 0.9546804 (200)	total: 13s	remaining: 4m 5s
400:	test: 0.9548215	best: 0.9548217 (393)	total: 25.9s	remaining: 3m 52s
600:	test: 0.9548475	best: 0.9548487 (585)	total: 38.8s	remaining: 3m 39s
800:	test: 0.9548419	best: 0.9548498 (621)	total: 51.8s	remaining: 3m 26s
bestTest = 0.9548498392
bestIteration = 621
Shrink model to first 622 iterations.
[cat_alt2] Fold 3/5


...

0:	test: 0.9448675	best: 0.9448675 (0)	total: 97.9ms	remaining: 6m 31s
200:	test: 0.9551292	best: 0.9551292 (200)	total: 13.3s	remaining: 4m 11s
400:	test: 0.9552705	best: 0.9552715 (391)	total: 26.6s	remaining: 3m 58s
600:	test: 0.9552905	best: 0.9552905 (600)	total: 39.6s	remaining: 3m 43s
800:	test: 0.9552899	best: 0.9552942 (718)	total: 52.6s	remaining: 3m 30s
bestTest = 0.9552941918
bestIteration = 718
Shrink model to first 719 iterations.
[cat_alt2] Fold 4/5


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9442605	best: 0.9442605 (0)	total: 102ms	remaining: 6m 47s
200:	test: 0.9559385	best: 0.9559385 (200)	total: 13.1s	remaining: 4m 6s
400:	test: 0.9560955	best: 0.9560955 (400)	total: 26.3s	remaining: 3m 55s
600:	test: 0.9561122	best: 0.9561125 (579)	total: 39.2s	remaining: 3m 41s
800:	test: 0.9561155	best: 0.9561179 (684)	total: 52.2s	remaining: 3m 28s
bestTest = 0.956117928
bestIteration = 684
Shrink model to first 685 iterations.
[cat_alt2] Fold 5/5


Default metric period is 5 because AUC is/are not implemented for GPU


0:	test: 0.9446786	best: 0.9446786 (0)	total: 99.4ms	remaining: 6m 37s
200:	test: 0.9550477	best: 0.9550477 (200)	total: 13.3s	remaining: 4m 11s
400:	test: 0.9552246	best: 0.9552254 (395)	total: 26.5s	remaining: 3m 57s
600:	test: 0.9552459	best: 0.9552476 (563)	total: 39.4s	remaining: 3m 42s
bestTest = 0.955247581
bestIteration = 563
Shrink model to first 564 iterations.
[cat_alt2][seed=2024] OOF AUC = 0.955592

[cat_alt2 ensemble] OOF AUC = 0.955614


### 8d. RealMLP

In [14]:
REALMLP_PARAMS = {
    'random_state': RANDOM_STATE,
    'verbosity': 2,
    'n_epochs': 100,
    'batch_size': 2**12,
    'n_ens': 8,
    'use_early_stopping': True,
    'early_stopping_additive_patience': 20,
    'early_stopping_multiplicative_patience': 1,
    'act': "mish",
    'embedding_size': 8,
    'first_layer_lr_factor': 0.5962121993798933,
    'val_metric_name': '1-auc_ovr',
    'hidden_sizes': "rectangular",
    'hidden_width': 384,
    'lr': 0.04,
    'ls_eps': 0.011498317194338772,
    'ls_eps_sched': "coslog4",
    'max_one_hot_cat_size': 18,
    'n_hidden_layers': 4,
    'p_drop': 0.07301419697186451,
    'p_drop_sched': "flat_cos",
    'plr_hidden_1': 16,
    'plr_hidden_2': 8,
    'plr_lr_factor': 0.1151437622270563,
    'plr_sigma': 2.3316811282666916,
    'scale_lr_factor': 2.244801835541429,
    'sq_mom': 1.0 - 0.011834054955582318,
    'wd': 0.02369230879235962,
    'device': DEVICE,
}

def make_realmlp(params: dict, cat_col_names: list[str]) -> RealMLP_TD_Classifier:
    """Instantiate RealMLP; falls back gracefully if cat_col_names is unsupported."""
    try:
        return RealMLP_TD_Classifier(**params, cat_col_names=cat_col_names)
    except TypeError:
        return RealMLP_TD_Classifier(**params)


oof_realmlp  = np.zeros(len(X_comp), dtype="float32")
test_realmlp = np.zeros(len(X_test_comp), dtype="float32")
fold_scores  = []

for fold, (train_idx, val_idx) in enumerate(skf.split(X_comp, y_comp), start=1):
    print(f"\n--- RealMLP Fold {fold}/{N_FOLDS} ---")
    X_tr, X_val = X_comp.iloc[train_idx], X_comp.iloc[val_idx]
    y_tr, y_val = y_comp.iloc[train_idx], y_comp.iloc[val_idx]

    model = make_realmlp(REALMLP_PARAMS, cat_cols)
    model.fit(X_tr, y_tr.values, X_val, y_val.values)

    val_probs  = model.predict_proba(X_val)[:, 1]
    test_probs = model.predict_proba(X_test_comp)[:, 1]

    oof_realmlp[val_idx]  = val_probs.astype("float32")
    test_realmlp         += (test_probs / N_FOLDS).astype("float32")

    score = roc_auc_score(y_val, val_probs)
    fold_scores.append(score)
    print(f"Fold {fold} AUC: {score:.6f}")

    if DEVICE == 'cuda':
        torch.cuda.empty_cache()

# Alias for downstream meta-feature construction
oof_realmlp_ens  = oof_realmlp.copy()
test_realmlp_ens = test_realmlp.copy()

print(f"\n[RealMLP] OOF AUC = {roc_auc_score(y_comp, oof_realmlp):.6f}")
print("Fold AUCs:", [f"{s:.4f}" for s in fold_scores])


--- RealMLP Fold 1/5 ---
Columns classified as continuous: []
Columns classified as categorical: ['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium']


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 1/100: val 1-auc_ovr = 0.047396
Epoch 2/100: val 1-auc_ovr = 0.044425
Epoch 3/100: val 1-auc_ovr = 0.043934
Epoch 4/100: val 1-auc_ovr = 0.043966
Epoch 5/100: val 1-auc_ovr = 0.043908
Epoch 6/100: val 1-auc_ovr = 0.043883
Epoch 7/100: val 1-auc_ovr = 0.043881
Epoch 8/100: val 1-auc_ovr = 0.043909
Epoch 9/100: val 1-auc_ovr = 0.043943
Epoch 10/100: val 1-auc_ovr = 0.043965
Epoch 11/100: val 1-auc_ovr = 0.043936
Epoch 12/100: val 1-auc_ovr = 0.043983
Epoch 13/100: val 1-auc_ovr = 0.043933
Epoch 14/100: val 1-auc_ovr = 0.043955
Epoch 15/100: val 1-auc_ovr = 0.043933
Epoch 16/100: val 1-auc_ovr = 0.043922
Epoch 17/100: val 1-auc_ovr = 0.043947
Epoch 18/100: val 1-auc_ovr = 0.043961
Epoch 19/100: val 1-auc_ovr = 0.043973
Epoch 20/100: val 1-auc_ovr = 0.043979
Epoch 21/100: val 1-auc_ovr = 0.043983
Epoch 22/100: val 1-auc_ovr = 0.043984
Epoch 23/100: val 1-auc_ovr = 0.044007
Epoch 24/100: val 1-auc_ovr = 0.044003
Epoch 25/100: val 1-auc_ovr = 0.043981
Epoch 26/100: val 1-auc_ovr = 0.04

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Fold 1 AUC: 0.956119

--- RealMLP Fold 2/5 ---
Columns classified as continuous: []
Columns classified as categorical: ['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium']


...

Fold 4 AUC: 0.955436

--- RealMLP Fold 5/5 ---
Columns classified as continuous: []
Columns classified as categorical: ['Age', 'Sex', 'Chest pain type', 'BP', 'Cholesterol', 'FBS over 120', 'EKG results', 'Max HR', 'Exercise angina', 'ST depression', 'Slope of ST', 'Number of vessels fluro', 'Thallium']


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Epoch 1/100: val 1-auc_ovr = 0.047628
Epoch 2/100: val 1-auc_ovr = 0.044322
Epoch 3/100: val 1-auc_ovr = 0.043898
Epoch 4/100: val 1-auc_ovr = 0.043818
Epoch 5/100: val 1-auc_ovr = 0.043801
Epoch 6/100: val 1-auc_ovr = 0.043785
Epoch 7/100: val 1-auc_ovr = 0.043774
Epoch 8/100: val 1-auc_ovr = 0.043792
Epoch 9/100: val 1-auc_ovr = 0.043812
Epoch 10/100: val 1-auc_ovr = 0.043804
Epoch 11/100: val 1-auc_ovr = 0.043865
Epoch 12/100: val 1-auc_ovr = 0.043801
Epoch 13/100: val 1-auc_ovr = 0.043811
Epoch 14/100: val 1-auc_ovr = 0.043784
Epoch 15/100: val 1-auc_ovr = 0.043829
Epoch 16/100: val 1-auc_ovr = 0.043795
Epoch 17/100: val 1-auc_ovr = 0.043799
Epoch 18/100: val 1-auc_ovr = 0.043831
Epoch 19/100: val 1-auc_ovr = 0.043840
Epoch 20/100: val 1-auc_ovr = 0.043843
Epoch 21/100: val 1-auc_ovr = 0.043851
Epoch 22/100: val 1-auc_ovr = 0.043857
Epoch 23/100: val 1-auc_ovr = 0.043858
Epoch 24/100: val 1-auc_ovr = 0.043841
Epoch 25/100: val 1-auc_ovr = 0.043835
Epoch 26/100: val 1-auc_ovr = 0.04

GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1]


Fold 5 AUC: 0.956226

[RealMLP] OOF AUC = 0.955657
Fold AUCs: ['0.9561', '0.9549', '0.9558', '0.9554', '0.9562']


### 8e. XGBoost

In [15]:
# XGBoost requires numeric inputs — convert category columns to integer codes
X_comp_xgb, X_test_xgb = to_xgb_matrix(X_comp, X_test_comp, cat_cols)

oof_xgb  = np.zeros(len(X_comp_xgb), dtype="float32")
test_xgb = np.zeros(len(X_test_xgb), dtype="float32")

for fold, (tr_idx, val_idx) in enumerate(skf.split(X_comp_xgb, y_comp), start=1):
    print(f"[XGB] Fold {fold}/{N_FOLDS}")
    X_tr, X_val = X_comp_xgb.iloc[tr_idx], X_comp_xgb.iloc[val_idx]
    y_tr, y_val = y_comp.iloc[tr_idx], y_comp.iloc[val_idx]

    model = xgb.XGBClassifier(
        n_estimators=5000, learning_rate=0.03, max_depth=4,
        min_child_weight=8, subsample=0.75, colsample_bytree=0.75,
        reg_alpha=0.5, reg_lambda=2.0, gamma=0.5,
        objective="binary:logistic", eval_metric="auc",
        tree_method="hist",
        random_state=RANDOM_STATE + fold, n_jobs=-1,
        device="gpu" if DEVICE == "cuda" else "cpu",
    )
    model.fit(X_tr, y_tr, eval_set=[(X_val, y_val)], verbose=200)

    oof_xgb[val_idx]  = model.predict_proba(X_val)[:, 1].astype("float32")
    test_xgb         += (model.predict_proba(X_test_xgb)[:, 1] / N_FOLDS).astype("float32")

xgb_oof_auc = roc_auc_score(y_comp, oof_xgb)
print(f"\n[XGB] OOF AUC = {xgb_oof_auc:.6f}")

[XGB] Fold 1/5
[0]	validation_0-auc:0.91974
[200]	validation_0-auc:0.95400
[400]	validation_0-auc:0.95485
[600]	validation_0-auc:0.95530
[800]	validation_0-auc:0.95556
[1000]	validation_0-auc:0.95570
[1200]	validation_0-auc:0.95576
[1400]	validation_0-auc:0.95580
[1600]	validation_0-auc:0.95583
[1800]	validation_0-auc:0.95583
[2000]	validation_0-auc:0.95583
[2200]	validation_0-auc:0.95582
[2400]	validation_0-auc:0.95581
[2600]	validation_0-auc:0.95579
[2800]	validation_0-auc:0.95578
[3000]	validation_0-auc:0.95577
[3200]	validation_0-auc:0.95576
[3400]	validation_0-auc:0.95574
[3600]	validation_0-auc:0.95572
[3800]	validation_0-auc:0.95570
[4000]	validation_0-auc:0.95569
[4200]	validation_0-auc:0.95567
[4400]	validation_0-auc:0.95565
[4600]	validation_0-auc:0.95563
[4800]	validation_0-auc:0.95561
[4999]	validation_0-auc:0.95559
[XGB] Fold 2/5
[0]	validation_0-auc:0.89475
[200]	validation_0-auc:0.95295
[400]	validation_0-auc:0.95395
[600]	validation_0-auc:0.95435
[800]	validation_0-auc:

### 8f. Base model summary

In [16]:
# Convenience aliases
oof_real  = oof_realmlp.copy()
test_real = test_realmlp.copy()

# Noise-feature placeholder (filled with zeros if not computed elsewhere)
oof_p_noise_feat  = safe_array(None, len(y_comp),       fill_value=0.0, name="oof_p_noise")
test_p_noise_feat = safe_array(None, len(X_test_comp),  fill_value=0.0, name="test_p_noise")

# Pairwise Spearman correlations (diversity diagnostic)
print("=== Base model OOF AUC ===")
for name, preds in [("CatBoost", oof_cat), ("CatBoost ALT", oof_cat_alt),
                    ("CatBoost ALT2", oof_cat_alt2), ("RealMLP", oof_realmlp), ("XGBoost", oof_xgb)]:
    print(f"  [{name:15s}] {roc_auc_score(y_comp, preds):.6f}")

print("\n=== Pairwise Spearman correlation ===")
pairs = [("cat", oof_cat), ("realmlp", oof_realmlp), ("xgb", oof_xgb)]
for i, (na, pa) in enumerate(pairs):
    for nb, pb in pairs[i+1:]:
        print(f"  {na} × {nb}: {spearmanr(pa, pb).correlation:.4f}")

=== Base model OOF AUC ===
  [CatBoost       ] 0.955611
  [CatBoost ALT   ] 0.955638
  [CatBoost ALT2  ] 0.955614
  [RealMLP        ] 0.955657
  [XGBoost        ] 0.955220

=== Pairwise Spearman correlation ===
  cat × realmlp: 0.9980
  cat × xgb: 0.9972
  realmlp × xgb: 0.9972


## 9. Meta Feature Construction

In [17]:
# Assemble base OOF/test dicts for meta-learning
base_cols = ["cat", "realmlp", "cat_alt2"]

meta_oof = pd.DataFrame({
    "id": comp_ids.values if hasattr(comp_ids, "values") else comp_ids,
    "y":  y_comp.values if hasattr(y_comp, "values") else y_comp,
    "cat":      np.asarray(oof_cat, dtype=np.float64),
    "realmlp":  np.asarray(oof_realmlp, dtype=np.float64),
    "cat_alt2": np.asarray(oof_cat_alt2, dtype=np.float64),
})

meta_test = pd.DataFrame({
    "id": test_ids.values if hasattr(test_ids, "values") else test_ids,
    "cat":      np.asarray(test_cat, dtype=np.float64),
    "realmlp":  np.asarray(test_realmlp, dtype=np.float64),
    "cat_alt2": np.asarray(test_cat_alt2, dtype=np.float64),
})

# Add rank / prob / z-score variants of each base model
for c in base_cols:
    meta_oof[f"{c}_rank"]  = rank01(meta_oof[c].values)
    meta_test[f"{c}_rank"] = rank01(meta_test[c].values)

    meta_oof[f"{c}_prob"]  = meta_oof[c].values
    meta_test[f"{c}_prob"] = meta_test[c].values

    meta_oof[f"{c}_z"]  = zscore(meta_oof[c].values)
    meta_test[f"{c}_z"] = zscore(meta_test[c].values)

# Quick sanity check
rank_mean_oof = meta_oof[[f"{c}_rank" for c in base_cols]].mean(axis=1).values
print("Reference blends:")
for c in base_cols:
    print(f"  [{c}] {roc_auc_score(meta_oof['y'], meta_oof[c]):.6f}")
print(f"  [rank mean]   {roc_auc_score(meta_oof['y'], rank_mean_oof):.6f}")

display(meta_oof.head())

Reference blends:
  [cat] 0.955611
  [realmlp] 0.955657
  [cat_alt2] 0.955614
  [rank mean]   0.955711


,id,y,cat,realmlp,cat_alt2,cat_rank,cat_prob,cat_z,realmlp_rank,realmlp_prob,realmlp_z,cat_alt2_rank,cat_alt2_prob,cat_alt2_z
0,0,1,0.997245,0.993483,0.997252,0.970660,0.997245,1.345807,0.939213,0.993483,1.338115,0.968293,0.997252,1.345356
1,1,0,0.010252,0.008669,0.010726,0.103453,0.010252,-1.072640,0.076782,0.008669,-1.081121,0.108580,0.010726,-1.072691
2,2,0,0.008682,0.009887,0.008372,0.087434,0.008682,-1.076487,0.089937,0.009887,-1.078130,0.084713,0.008372,-1.078462
3,3,0,0.048835,0.048276,0.043225,0.289837,0.048835,-0.978101,0.287318,0.048276,-0.983825,0.274279,0.043225,-0.993033
4,4,1,0.996725,0.997424,0.997380,0.964437,0.996725,1.344532,0.980110,0.997424,1.347796,0.969829,0.997380,1.345668


## 10. Meta Learning (Ridge)

### 10a. Compare feature modes (rank / prob / rank+prob / rank+z)

In [18]:
oof_dict  = {c: meta_oof[c].values   for c in base_cols}
test_dict = {c: meta_test[c].values  for c in base_cols}
model_order = base_cols

# Meta Ridge diagnostic (rank / rank+prob)
META_BASE_COLS = ["cat", "realmlp", "cat_alt2"]
oof_dict_meta  = {c: meta_oof[c].values  for c in META_BASE_COLS}
test_dict_meta = {c: meta_test[c].values for c in META_BASE_COLS}
model_order_meta = META_BASE_COLS

results_meta = {}
for feature_mode in ["rank", "rank+prob"]:
    print(f"feature_mode: {feature_mode} (META_BASE_COLS={META_BASE_COLS})")
    results_meta[feature_mode] = fit_meta_ridge(
        y=y_comp.values,
        oof_dict=oof_dict_meta, test_dict=test_dict_meta,
        model_order=model_order_meta,
        feature_mode=feature_mode,
        n_splits=N_FOLDS, seed=RANDOM_STATE,
    )

best_mode = max(results_meta, key=lambda k: results_meta[k]["meta_oof_auc"])
print(f"Best meta feature mode: {best_mode}  AUC={results_meta[best_mode]['meta_oof_auc']:.6f}")


feature_mode: rank (META_BASE_COLS=['cat', 'realmlp', 'cat_alt2'])
  [Fold 1] AUC=0.956144, alpha=30.0
  [Fold 2] AUC=0.954984, alpha=30.0
  [Fold 3] AUC=0.955813, alpha=10.0
  [Fold 4] AUC=0.955438, alpha=10.0
  [Fold 5] AUC=0.956277, alpha=30.0
  [OOF AUC] 0.955719  (feature_mode=rank)
feature_mode: rank+prob (META_BASE_COLS=['cat', 'realmlp', 'cat_alt2'])
  [Fold 1] AUC=0.956151, alpha=10.0
  [Fold 2] AUC=0.954983, alpha=10.0
  [Fold 3] AUC=0.955814, alpha=30.0
  [Fold 4] AUC=0.955453, alpha=10.0
  [Fold 5] AUC=0.956279, alpha=30.0
  [OOF AUC] 0.955697  (feature_mode=rank+prob)
Best meta feature mode: rank  AUC=0.955719


### 10b. Quick comparison with simple blends

In [19]:
oof_simple = 0.9 * oof_cat + 0.1 * oof_real

rank_mean_2models = np.mean([rank01(meta_oof[c].values) for c in ["cat", "realmlp"]], axis=0)
rank_mean_3models = np.mean([rank01(oof_dict[m]) for m in model_order], axis=0)

print("=== Ensemble OOF AUC comparison ===")
print(f"  Cat only            : {roc_auc_score(y_comp, oof_cat):.6f}")
print(f"  RealMLP only        : {roc_auc_score(y_comp, oof_real):.6f}")
print(f"  XGB only            : {roc_auc_score(y_comp, oof_xgb):.6f}")
print(f"  Cat alt2 only       : {roc_auc_score(y_comp, oof_cat_alt2):.6f}")
print(f"  Simple 0.9/0.1      : {roc_auc_score(y_comp, oof_simple):.6f}")
print(f"  Rank mean (2)       : {roc_auc_score(y_comp, rank_mean_2models):.6f}")
print(f"  Rank mean (3)       : {roc_auc_score(y_comp, rank_mean_3models):.6f}")
print(f"  Meta Ridge (rank-2) : {results_meta[best_mode]['meta_oof_auc']:.6f}  [{best_mode}]")


=== Ensemble OOF AUC comparison ===
  Cat only            : 0.955611
  RealMLP only        : 0.955657
  XGB only            : 0.955220
  Cat alt2 only       : 0.955614
  Simple 0.9/0.1      : 0.955645
  Rank mean (2)       : 0.955719
  Rank mean (3)       : 0.955711
  Meta Ridge (rank-2) : 0.955719  [rank]


## 11. Repeated CV Comparison (3 seeds)

Robust comparison of Cat only / Rank mean / Meta Ridge across multiple CV seeds and alpha values.

In [20]:
REPEATED_SEEDS = CFG['repeated_seeds']
ALPHAS = CFG['ridge_alphas']

META_BASE_COLS = ["cat", "realmlp", "cat_alt2"]
FEATURE_SETS = {
    "rank_only_2models": ["cat_rank", "realmlp_rank"],
    "rank_prob_2models": ["cat_rank", "realmlp_rank", "cat_prob", "realmlp_prob"],
    "rank_only_3models": ["cat_rank", "realmlp_rank", "cat_alt2_rank"],
    "rank_prob_3models": ["cat_rank", "realmlp_rank", "cat_alt2_rank",
                           "cat_prob", "realmlp_prob", "cat_alt2_prob"],
}

y_all        = meta_oof["y"].values.astype(int)
oof_cat_base = meta_oof["cat"].values.astype(np.float64)
test_cat_base= meta_test["cat"].values.astype(np.float64)

oof_rank_mean_2models  = meta_oof[[f"{c}_rank" for c in ["cat", "realmlp"]]].mean(axis=1).values.astype(np.float64)
test_rank_mean_2models = meta_test[[f"{c}_rank" for c in ["cat", "realmlp"]]].mean(axis=1).values.astype(np.float64)

oof_rank_mean_3models  = meta_oof[[f"{c}_rank" for c in META_BASE_COLS]].mean(axis=1).values.astype(np.float64)
test_rank_mean_3models = meta_test[[f"{c}_rank" for c in META_BASE_COLS]].mean(axis=1).values.astype(np.float64)

rows = []
best_meta_cfg   = None
best_meta_score = -1.0

for seed in REPEATED_SEEDS:
    print(f"{'='*60}\nRepeated CV seed = {seed}")
    skf_rep = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)

    # Baselines (use full OOF directly — seed only affects meta-Ridge fold split)
    rows.append({"rep_seed": seed, "method": "cat_only",  "feature_set": "-",
                 "alpha": np.nan, "cv_auc_mean": roc_auc_score(y_all, oof_cat_base)})
    rows.append({"rep_seed": seed, "method": "rank_mean_2models", "feature_set": "rank_only_2models",
                 "alpha": np.nan, "cv_auc_mean": roc_auc_score(y_all, oof_rank_mean_2models)})
    rows.append({"rep_seed": seed, "method": "rank_mean_3models", "feature_set": "rank_only_3models",
                 "alpha": np.nan, "cv_auc_mean": roc_auc_score(y_all, oof_rank_mean_3models)})

    # Meta Ridge sweep
    for fs_name, fs_cols in FEATURE_SETS.items():
        X_meta      = meta_oof[fs_cols].values.astype(np.float64)
        X_meta_test = meta_test[fs_cols].values.astype(np.float64)

        for alpha in ALPHAS:
            meta_oof_pred  = np.zeros(len(meta_oof), dtype=np.float64)
            meta_test_pred = np.zeros(len(meta_test), dtype=np.float64)

            for fold, (tr_idx, va_idx) in enumerate(skf_rep.split(X_meta, y_all), start=1):
                reg = Ridge(alpha=alpha, fit_intercept=True, random_state=seed)
                reg.fit(X_meta[tr_idx], y_all[tr_idx])
                meta_oof_pred[va_idx]  = reg.predict(X_meta[va_idx])
                meta_test_pred        += reg.predict(X_meta_test) / N_FOLDS

            auc_meta = roc_auc_score(y_all, meta_oof_pred)
            rows.append({"rep_seed": seed, "method": "meta_ridge",
                         "feature_set": fs_name, "alpha": alpha, "cv_auc_mean": auc_meta})

            if auc_meta > best_meta_score:
                best_meta_score = auc_meta
                best_meta_cfg   = {"rep_seed": seed, "feature_set": fs_name, "alpha": alpha,
                                   "oof_pred": meta_oof_pred.copy(), "test_pred": meta_test_pred.copy(),
                                   "fs_cols": fs_cols}

df_rep = pd.DataFrame(rows)

print("===== Method summary (top 20 by mean AUC) =====")
display(
    df_rep.groupby(["method", "feature_set", "alpha"], dropna=False)["cv_auc_mean"]
    .agg(["mean", "std", "min", "max", "count"])
    .sort_values("mean", ascending=False)
    .head(20)
)

print(f"Best single-seed Meta Ridge: feature_set={best_meta_cfg['feature_set']}"
      f"alpha={best_meta_cfg['alpha']}, seed={best_meta_cfg['rep_seed']}, AUC={best_meta_score:.6f}")


Repeated CV seed = 42
Repeated CV seed = 2024
Repeated CV seed = 2025
===== Method summary (top 20 by mean AUC) =====


mean       std       min  \
method            feature_set       alpha                                 
meta_ridge        rank_only_3models 10.0   0.955722  0.000003  0.955718   
                                    30.0   0.955722  0.000002  0.955719   
                                    100.0  0.955719  0.000002  0.955717   
rank_mean_2models rank_only_2models NaN    0.955719  0.000000  0.955719   
meta_ridge        rank_only_2models 10.0   0.955718  0.000002  0.955715   
                                    30.0   0.955718  0.000002  0.955715   
                                    100.0  0.955717  0.000003  0.955714   
                                    300.0  0.955717  0.000003  0.955714   
                  rank_only_3models 300.0  0.955715  0.000001  0.955714   
                  rank_prob_2models 300.0  0.955713  0.000004  0.955708   
                  rank_prob_3models 10.0   0.955712  0.000014  0.955697   
                  rank_prob_2models 100.0  0.955712  0.000007  0.955704   
                  rank_prob_3models 300.0  0.955711  0.000004  0.955707   
rank_mean_3models rank_only_3models NaN    0.955711  0.000000  0.955711   
meta_ridge        rank_prob_3models 30.0   0.955710  0.000012  0.955696   
                                    100.0  0.955710  0.000008  0.955701   
                  rank_prob_2models 30.0   0.955710  0.000010  0.955698   
                                    10.0   0.955710  0.000013  0.955695   
cat_only          -                 NaN    0.955611  0.000000  0.955611   

                                                max  count  
method            feature_set       alpha                   
meta_ridge        rank_only_3models 10.0   0.955724      3  
                                    30.0   0.955723      3  
                                    100.0  0.955720      3  
rank_mean_2models rank_only_2models NaN    0.955719      3  
meta_ridge        rank_only_2models 10.0   0.955719      3  
                                    30.0   0.955719      3  
                                    100.0  0.955719      3  
                                    300.0  0.955719      3  
                  rank_only_3models 300.0  0.955716      3  
                  rank_prob_2models 300.0  0.955716      3  
                  rank_prob_3models 10.0   0.955720      3  
                  rank_prob_2models 100.0  0.955716      3  
                  rank_prob_3models 300.0  0.955714      3  
rank_mean_3models rank_only_3models NaN    0.955711      3  
meta_ridge        rank_prob_3models 30.0   0.955718      3  
                                    100.0  0.955715      3  
                  rank_prob_2models 30.0   0.955716      3  
                                    10.0   0.955717      3  
cat_only          -                 NaN    0.955611      3

Best single-seed Meta Ridge: feature_set=rank_only_3modelsalpha=10.0, seed=2024, AUC=0.955724


In [21]:
# ── Select best config by average across repeated seeds ────────────────────────
meta_summary = (
    df_rep[df_rep["method"] == "meta_ridge"]
    .groupby(["feature_set", "alpha"])["cv_auc_mean"]
    .agg(["mean", "std", "count"])
    .reset_index()
    .sort_values(["mean", "std"], ascending=[False, True])
)
display(meta_summary.head(20))

best_fs       = meta_summary.iloc[0]["feature_set"]
best_alpha    = float(meta_summary.iloc[0]["alpha"])
best_fs_cols  = FEATURE_SETS[best_fs]
print(f"[Selected] feature_set={best_fs}, alpha={best_alpha}")

# ── Re-fit with all repeated seeds for stable final predictions ────────────────
X_meta_sel      = meta_oof[best_fs_cols].values.astype(np.float64)
X_meta_test_sel = meta_test[best_fs_cols].values.astype(np.float64)

oof_meta_ridge_rep  = np.zeros(len(meta_oof),  dtype=np.float64)
test_meta_ridge_rep = np.zeros(len(meta_test), dtype=np.float64)

for seed in REPEATED_SEEDS:
    skf_rep = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=seed)
    oof_tmp  = np.zeros(len(meta_oof),  dtype=np.float64)
    test_tmp = np.zeros(len(meta_test), dtype=np.float64)

    for fold, (tr_idx, va_idx) in enumerate(skf_rep.split(X_meta_sel, y_all), start=1):
        reg = Ridge(alpha=best_alpha, fit_intercept=True, random_state=seed)
        reg.fit(X_meta_sel[tr_idx], y_all[tr_idx])
        oof_tmp[va_idx]  = reg.predict(X_meta_sel[va_idx])
        test_tmp        += reg.predict(X_meta_test_sel) / N_FOLDS

    oof_meta_ridge_rep  += oof_tmp  / len(REPEATED_SEEDS)
    test_meta_ridge_rep += test_tmp / len(REPEATED_SEEDS)

print("===== Final candidate OOF AUCs =====")
print(f"  Cat only             : {roc_auc_score(y_all, oof_cat_base):.6f}")
print(f"  Rank mean (2 models) : {roc_auc_score(y_all, oof_rank_mean_2models):.6f}")
print(f"  Rank mean (3 models) : {roc_auc_score(y_all, oof_rank_mean_3models):.6f}")
print(f"  Meta Ridge (best)    : {roc_auc_score(y_all, oof_meta_ridge_rep):.6f}")

# Collect candidates for submission
submission_candidates = {
    "cat_only":            test_cat_base.copy(),
    "rank_mean_2models":   test_rank_mean_2models.copy(),
    "rank_mean_3models":   test_rank_mean_3models.copy(),
    "meta_ridge_rep":      test_meta_ridge_rep.copy(),
}
oof_candidates = {
    "cat_only":            oof_cat_base.copy(),
    "rank_mean_2models":   oof_rank_mean_2models.copy(),
    "rank_mean_3models":   oof_rank_mean_3models.copy(),
    "meta_ridge_rep":      oof_meta_ridge_rep.copy(),
}

# Manual or auto selection
# selected_submission_name = "meta_ridge_rep"
selected_submission_name = max(
    oof_candidates.keys(),
    key=lambda k: roc_auc_score(y_comp, oof_candidates[k])
)
print("Selected candidate:", selected_submission_name)


,feature_set,alpha,mean,std,count
4,rank_only_3models,10.0,0.955722,0.000003,3
5,rank_only_3models,30.0,0.955722,0.000002,3
6,rank_only_3models,100.0,0.955719,0.000002,3
0,rank_only_2models,10.0,0.955718,0.000002,3
1,rank_only_2models,30.0,0.955718,0.000002,3
2,rank_only_2models,100.0,0.955717,0.000003,3
3,rank_only_2models,300.0,0.955717,0.000003,3
7,rank_only_3models,300.0,0.955715,0.000001,3
11,rank_prob_2models,300.0,0.955713,0.000004,3
12,rank_prob_3models,10.0,0.955712,0.000014,3


[Selected] feature_set=rank_only_3models, alpha=10.0
===== Final candidate OOF AUCs =====
  Cat only             : 0.955611
  Rank mean (2 models) : 0.955719
  Rank mean (3 models) : 0.955711
  Meta Ridge (best)    : 0.955723
Selected candidate: meta_ridge_rep


## 12. Submission Generation

In [22]:
# ── Validate candidates exist ──────────────────────────────────────────────────
for k in submission_candidates:
    assert k in oof_candidates, f"oof_candidates missing key: {k}"

# ── Save OOF comparison CSV (for analysis) ─────────────────────────────────────
oof_compare_df = pd.DataFrame({
    "id":            comp_ids.values if hasattr(comp_ids, "values") else comp_ids,
    "y_true":        y_comp.values if hasattr(y_comp, "values") else y_comp,
    "cat_only":      oof_candidates["cat_only"],
    "rank_mean_2models": oof_candidates["rank_mean_2models"],
    "rank_mean_3models": oof_candidates["rank_mean_3models"],
    "meta_ridge_rep":    oof_candidates["meta_ridge_rep"],
})
oof_compare_df.to_csv("oof_compare_candidates.csv", index=False)
print("Saved: oof_compare_candidates.csv")

# ── Save one submission file per candidate ─────────────────────────────────────
for name, pred in submission_candidates.items():
    sub_df = pd.DataFrame({
        "id": test_ids.values if hasattr(test_ids, "values") else test_ids,
        "Heart Disease": np.clip(pred.astype(np.float64), 0.0, 1.0),
        # Note: Ridge can produce values slightly outside [0,1]; clipping preserves AUC ranking.
    })
    fname = f"submission_{name}.csv"
    sub_df.to_csv(fname, index=False)
    print(f"Saved: {fname}  shape={sub_df.shape}")

# ── Final submission (selected candidate) ─────────────────────────────────────
final_pred = np.clip(submission_candidates[selected_submission_name].astype(np.float64), 0.0, 1.0)
submission = pd.DataFrame({
    "id": test_ids.values if hasattr(test_ids, "values") else test_ids,
    "Heart Disease": final_pred,
})
submission.to_csv("submission.csv", index=False)
print(f"Saved final: submission.csv  ({selected_submission_name})")

# ── Final OOF AUC summary ──────────────────────────────────────────────────────
print("OOF AUC summary:")
for name in ["cat_only", "rank_mean_2models", "rank_mean_3models", "meta_ridge_rep"]:
    auc = roc_auc_score(y_comp, oof_candidates[name])
    marker = " ← selected" if name == selected_submission_name else ""
    print(f"  [{name}] {auc:.6f}{marker}")


Saved: oof_compare_candidates.csv
Saved: submission_cat_only.csv  shape=(270000, 2)
Saved: submission_rank_mean_2models.csv  shape=(270000, 2)
Saved: submission_rank_mean_3models.csv  shape=(270000, 2)
Saved: submission_meta_ridge_rep.csv  shape=(270000, 2)
Saved final: submission.csv  (meta_ridge_rep)
OOF AUC summary:
  [cat_only] 0.955611
  [rank_mean_2models] 0.955719
  [rank_mean_3models] 0.955711
  [meta_ridge_rep] 0.955723 ← selected
